In [1]:
import torch
import soundfile as sf
from torchaudio import transforms as T
from neucodec import NeuCodec

model = NeuCodec.from_pretrained("neuphonic/neucodec")
model.eval()

# Load audio
y, sr = sf.read(
    "D:/Personal Projects/tune-ml/datasets/audio/audio/5061 Philippe Van Mullem - Canopy (Progress Mix).mp3"
)

y = torch.tensor(y).float()

# ---- FIX 1: stereo -> mono ----
if y.ndim == 1:
    y = y.unsqueeze(0)  # (1, T)
else:
    y = y.transpose(0, 1)  # (C, T)
    y = y.mean(dim=0, keepdim=True)  # (1, T)

# ---- FIX 2: resample ----
if sr != 16000:
    y = T.Resample(sr, 16000)(y)

# ---- FIX 3: batch dimension ----
y = y.unsqueeze(0)  # (B, 1, T)

print("Final shape:", y.shape)  # should be (1, 1, T)

with torch.no_grad():
    fsq_codes = model.encode_code(y)
    recon = model.decode_code(fsq_codes).cpu()

# save
sf.write("original.wav", y[0, 0].numpy(), 16000)
sf.write("reconstructed.wav", recon[0, 0].numpy(), 24000)

d:\Personal Projects\tune-ml\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Personal Projects\tune-ml\.venv\lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 1.22.4)
  from scipy.sparse import csr_matrix, issparse
W0514 12:20:00.557000 13080 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
d:\Personal Projects\tune-ml\.venv\lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
Loading weights: 100%|██████████| 773/773 [00:00<00:00, 5049.78it/s]
d:\

Final shape: torch.Size([1, 1, 1920000])


: 